[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/corrections/seance2_correction.ipynb)

# Séance 2.2 — Nettoyer des données réelles

**Correction** · durée : 2h — cinq defauts, chacun suivi de deux exercices

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- repérer les six défauts classiques d'un fichier réel
- convertir du texte en nombres et en dates
- traiter les valeurs manquantes en connaissance de cause
- supprimer les doublons et écarter les valeurs aberrantes
- classer des lignes selon une règle, sur la colonne entière
- construire un pipeline de nettoyage qu'on peut rejouer

## La séance précédente vous a menti

`ventes.csv` était **impeccable** : pas un trou, pas un doublon, des types
corrects. Ça n'arrive jamais.

Voici le même détaillant, mais l'export tel qu'il sort vraiment du système :
`ventes_sale.csv`.

> 🎯 **Votre mission de la séance :** transformer ce fichier en données
> exploitables, et savoir dire **combien de lignes** vous avez perdues au
> passage et **pourquoi**.

> 📋 **Comment on travaille.** Cinq défauts. Pour chacun : une démonstration
> sur une table appelée `propre`, puis **deux exercices** où vous nettoyez la
> vôtre, appelée `net` — un à trous, un que vous écrivez en entier. À la fin,
> le fichier propre, c'est vous qui l'aurez construit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

print(sale.shape)   ## (lignes, colonnes)
sale.head(5)        ## cinq lignes suffisent a reperer l'essentiel

Prenez 30 secondes pour regarder ces cinq lignes. Qu'est-ce qui cloche ?

In [ ]:
sale.info()   ## regarder surtout la colonne Dtype

### Le diagnostic

`info()` révèle déjà deux problèmes graves :

- **`prix` est de type `object`** — c'est du **texte**, pas un nombre. On ne
  peut donc rien calculer avec. Les coupables sont visibles dès les cinq
  premières lignes : le suffixe de `0,42 EUR`, et la **virgule** décimale de
  `3,75` là où Python attend un point.
- **`date` est de type `object`** — du texte aussi. Impossible de demander
  « quel mois ? ».

Et un troisième, visible sur le `non-null` :

- **`client_id` a des trous.**

Il y en a quatre autres qu'`info()` ne montre pas. On va les débusquer.

## Défaut 1 — Les doublons

**On commence toujours par là**, avant toute autre étape de nettoyage.

La raison est un problème de comptage. Imaginez que vous commenciez par
retirer les ventes sans client : vous notez « 407 lignes retirées ». Mais
parmi ces 407, certaines étaient des copies l'une de l'autre. Vous n'avez donc
pas retiré 407 ventes, vous en avez retiré moins — et vous ne saurez jamais
combien. En dédoublonnant d'abord, chaque ligne du fichier est une vente
distincte, et tous les comptes qui suivent veulent dire quelque chose.

`duplicated()` ne rend pas un nombre. Comme la comparaison `prix > 50` de la
séance 2.1, il pose une question **à chaque ligne** — « celle-ci, je l'ai déjà
vue plus haut ? » — et rend **une réponse `True` ou `False` par ligne**, soit
5 370 réponses ici. Pour en tirer un nombre, on compte les `True` avec
`.sum()`.

In [ ]:
marques = sale.duplicated()   ## True = ligne deja vue plus haut

print(marques.iloc[140:145])   ## la 143e est une copie d'une ligne d'avant
print("lignes strictement identiques :", marques.sum())   ## .sum() compte les True

Quatre `False` et un `True` : la ligne d'étiquette 143 est le premier doublon
du fichier. `.sum()` en trouve **251** au total.

251 lignes en double. Un client ne passe pas deux fois exactement la même
commande, à la même seconde, pour le même produit : c'est un **bug d'export**.

> ⚠️ Nuance importante : ici les lignes sont **strictement identiques sur
> toutes les colonnes**, donc on peut supprimer. Si seul le `cmd_id` était en
> double, ce pourrait être un vrai client commandant deux fois le même
> article. Vérifiez toujours **sur quelles colonnes** porte le doublon avant
> de supprimer.

In [ ]:
avant = len(sale)                        ## on note le point de depart
propre = sale.drop_duplicates().copy()   ## .copy() : un vrai tableau a soi

print(avant, "->", len(propre))          ## toujours mesurer ce qu'on retire

> 💡 Le `.copy()` dit à pandas : « fais-moi un vrai tableau indépendant ».
> Sans lui, il vous avertira plus tard que vous modifiez peut-être une simple
> vue du tableau d'origine. Prenez l'habitude de l'ajouter après un filtrage.

---

### ✏️ À vous 1a — Dédoublonner (toujours en premier)

> **Votre mission :**
> - À votre tour. Créer `net` : `sale` sans les lignes dupliquées.
> - `net` est **votre** table : c'est elle que vous nettoierez tout au long de la séance.
> - ⚠️ On dédoublonne **avant tout le reste**.

In [ ]:
# On dedoublonne AVANT tout le reste : sinon les doublons
# se propagent dans toutes les etapes suivantes.
# .copy() evite les avertissements quand on modifiera net plus bas.
net = sale.drop_duplicates().copy()   ## 5370 -> 5119 lignes

print(len(net))

In [ ]:
verifier("1a - apres dedoublonnage", len(net) == 5119,
         "5370 - 251 : la methode s'appelle drop_duplicates()")

---

### ✏️ À vous 1b — Le doublon qui n'en a pas l'air

> **Votre mission :**
> - Compter les doublons **exacts** de `sale` → `dbl_exacts`, puis les doublons sur le seul couple `cmd_id` + `prod_id` → `dbl_partiels`.
> - Les deux nombres diffèrent. Affichez quelques-unes des lignes concernées : qu'est-ce qui les distingue ?
> - *Nouveau :* `df.duplicated(subset=['a', 'b'])` ne compare que les colonnes nommées.

In [ ]:
dbl_exacts = sale.duplicated().sum()
dbl_partiels = sale.duplicated(subset=["cmd_id", "prod_id"]).sum()
print(dbl_exacts, "exacts |", dbl_partiels, "sur commande + produit")

# keep=False marque TOUTES les copies, ~ inverse la condition
partiels = sale[sale.duplicated(subset=["cmd_id", "prod_id"], keep=False)
                & ~sale.duplicated(keep=False)]
partiels[["cmd_id", "prod_id", "qte", "prix"]].head(6)

# Le meme achat saisi deux fois avec deux ecritures du prix
# ("8,50 EUR" et "8,50"). drop_duplicates() ne les voit pas : pour lui
# ce sont deux textes differents. Nettoyer le prix AVANT de dedoublonner
# en attraperait cinq de plus.

In [ ]:
verifier("1b - doublons exacts", dbl_exacts == 251,
         "duplicated() sans argument compare toutes les colonnes")
verifier("1c - doublons partiels", dbl_partiels == 256,
         "duplicated(subset=['cmd_id', 'prod_id']) ne compare que ces deux colonnes")

## Défaut 2 — Les valeurs manquantes

La commande à taper devant n'importe quel fichier :

In [ ]:
propre.isna().sum()   ## un compte de trous, colonne par colonne

407 lignes sans `client_id`. **Que faire ?**

Il n'y a pas de réponse universelle. Il y a une question à se poser :
**pourquoi cette valeur manque-t-elle ?**

Ici, probablement des ventes sans compte client (achat en magasin, commande
invitée). Donc :

| Votre question | La bonne décision |
|---|---|
| « Combien mes clients dépensent-ils ? » | **Supprimer** ces lignes : elles n'ont pas de client |
| « Quel est mon chiffre d'affaires total ? » | **Les garder** : ce sont de vraies ventes, les retirer fausserait le total |

### À quoi ressemblerait `fillna`, concrètement

`fillna(valeur)` remplace chaque trou par la valeur qu'on lui donne. Sur
cette colonne, ça s'écrirait comme ceci — regardez le résultat avant de
trouver la commande pratique.

In [ ]:
# On ecrit dans une colonne A COTE, jamais par-dessus l'originale
propre["client_id_new"] = propre["client_id"].fillna(0)   ## 0 dans les trous

print("trous restants :", propre["client_id_new"].isna().sum())
print(propre["client_id_new"].value_counts().head(3))

Plus un seul trou : mission accomplie ? Regardez le classement. Le « client
0 » arrive **deuxième du fichier** avec 407 achats, derrière un seul client
réel. Sauf que ce client n'existe pas : ce sont 407 acheteurs différents
regroupés sous une étiquette inventée. Toute analyse par client sera fausse,
et absolument rien ne vous préviendra.

> ⚠️ **Le piège à ne jamais commettre :** `fillna(0)` sur un identifiant.
> Remplir une valeur manquante, c'est **inventer une donnée** — ne le faites
> que si vous pouvez le justifier.

`fillna` a pourtant des usages parfaitement légitimes : une quantité absente
qu'on sait valoir 0, un libellé vide qu'on remplace par `"inconnu"`, un prix
manquant qu'on remplace par la médiane de sa catégorie. La question n'est
jamais « est-ce que ça marche ? » mais « qu'est-ce que j'affirme en
remplissant ce trou ? ».

In [ ]:
# Celle-la ne nous sert a rien : on la retire avant de continuer
propre = propre.drop(columns=["client_id_new"])   ## drop(columns=[...])

In [ ]:
# Notre question portera sur les clients : on supprime ces lignes,
# mais on note combien on en perd.
avant = len(propre)
propre = propre.dropna(subset=["client_id"]).copy()   ## cette colonne seule

print(avant, "->", len(propre), f"({avant - len(propre)} lignes retirees)")

---

### ✏️ À vous 2a — Écarter les ventes sans client

> **Votre mission :**
> - Retirer de `net` les lignes dont `client_id` est manquant.
> - Rappel : on le fait parce que notre question porte sur les **clients**. Pour une question sur le chiffre d'affaires total, ce serait une erreur.

In [ ]:
# subset=["client_id"] : on ne supprime que si CETTE colonne est vide,
# pas des qu'une colonne quelconque a un trou
net = net.dropna(subset=["client_id"]).copy()   ## un dropna() nu en ferait plus

print(len(net))

In [ ]:
verifier("2a - lignes avec client", len(net) == 4712,
         "dropna(subset=[...]) cible une colonne precise")

---

### ✏️ À vous 2b — Le diagnostic, en pourcentage

> **Votre mission :**
> - Sur le fichier **brut** `sale` : le taux de valeurs manquantes de chaque colonne, en %, arrondi à 2 décimales → `taux_na`.
> - Combien de colonnes sont réellement touchées ? → `nb_touchees`
> - *Nouveau :* sur des True/False, `.mean()` donne directement une proportion — `df.isna().mean()`.

In [ ]:
# isna() donne True/False, mean() en fait une proportion, x100 un %
taux_na = (sale.isna().mean() * 100).round(2)
nb_touchees = (taux_na > 0).sum()

print(taux_na)
print(nb_touchees, "colonne(s) touchee(s)")

# Une seule colonne est touchee : client_id, a 7,99 %. Un diagnostic
# tient en une ligne, et il dit ou porter l'effort.

In [ ]:
verifier("2b - colonnes touchees", nb_touchees == 1,
         "isna().mean() donne une part par colonne : comptez celles > 0")
verifier("2c - taux de client_id", taux_na["client_id"] == 7.99,
         "multipliez la proportion par 100, puis round(2)")

## Défaut 3 — Des nombres stockés en texte

C'est le défaut le plus courant, et le plus sournois.

In [ ]:
propre["prix"].head(4)   ## du texte, pas des nombres

Deux problèmes dans une seule colonne :

1. Le suffixe **` EUR`** sur certaines valeurs.
2. La **virgule** comme séparateur décimal — convention française, alors que
   Python attend un point.

Trois étapes, dans cet ordre :

In [ ]:
# .str donne acces aux operations sur du texte, colonne entiere d'un coup
prix_txt = propre["prix"].str.replace(" EUR", "")   ## 1. l'unite
prix_txt = prix_txt.str.replace(",", ".")           ## 2. la virgule

propre["prix"] = pd.to_numeric(prix_txt, errors="coerce")   ## 3. la conversion
propre["prix"].head(4)   ## le type a change : ce sont des nombres

**`errors="coerce"`** veut dire : *« si tu n'arrives pas à convertir une
valeur, mets `NaN` au lieu de tout faire planter »*. C'est très pratique —
et très dangereux si on ne vérifie pas ensuite.

In [ ]:
# Reflexe obligatoire apres un coerce : combien de valeurs ont ete perdues ?
print("prix non convertis :", propre["prix"].isna().sum())

Zéro. Notre conversion est propre. **Faites systématiquement cette
vérification** : sans elle, vous pourriez transformer silencieusement 3 000
prix en `NaN` et ne vous en apercevoir qu'en présentant vos résultats.

---

### ✏️ À vous 3a — Le prix en nombre

> **Votre mission :**
> - Sur `net` : enlever le suffixe ` EUR`, remplacer la virgule par un point, convertir en nombre, et remettre le résultat dans `net['prix']`.
> - Puis vérifier qu'aucune valeur n'a été perdue → `nb_prix_perdus`.

In [ ]:
# Etape 1 : enlever le suffixe " EUR"
txt = net["prix"].str.replace(" EUR", "")   ## .str = du texte

# Etape 2 : virgule francaise -> point, que Python comprend
txt = txt.str.replace(",", ".")

# Etape 3 : conversion. errors="coerce" met NaN au lieu de planter...
net["prix"] = pd.to_numeric(txt, errors="coerce")

# ... donc on verifie tout de suite combien de NaN ont ete crees
nb_prix_perdus = net["prix"].isna().sum()   ## doit valoir 0

print(net["prix"].dtype, "|", nb_prix_perdus, "valeurs perdues")

In [ ]:
verifier("3a - prix numerique", net["prix"].dtype == "float64",
         "pd.to_numeric convertit une colonne texte en nombres")
verifier("3b - aucune perte", nb_prix_perdus == 0,
         "si > 0, c'est qu'il reste du texte non converti dans la colonne")

---

### ✏️ À vous 3b — Regarder avant de convertir

> **Votre mission :**
> - Sur `sale` : convertir `prix` en nombre avec `errors='coerce'` en ne remplaçant **que** la virgule, et **sans écraser la colonne d'origine** → `essai`.
> - Combien de valeurs échouent ? → `nb_refus`. Puis **affichez quelques-unes des valeurs fautives d'origine**.
> - Que se serait-il passé si on avait enchaîné sur un `dropna()` sans regarder ?

In [ ]:
essai = pd.to_numeric(sale["prix"].astype(str).str.replace(",", "."),
                      errors="coerce")
nb_refus = essai.isna().sum()

print(nb_refus, "valeurs refusent la conversion")
sale.loc[essai.isna(), "prix"].head(5)   ## REGARDER les fautives d'origine

# Ce ne sont pas des erreurs : c'est un suffixe " EUR". Un coerce
# aveugle aurait mis 798 prix a NaN, puis un dropna les aurait
# supprimes — 15 % du fichier perdu pour une unite ecrite en toutes
# lettres.

In [ ]:
verifier("3c - valeurs refusant la conversion", nb_refus == 798,
         "remplacez seulement la virgule : le suffixe EUR reste, et il bloque")

## Défaut 4 — Les dates

Le plus piégeux. On procède en trois temps — mais d'abord, une question de
méthode.

### Temps 0 : savoir ce qu'on attend

On ne peut pas juger si une conversion a réussi sans savoir à quoi devrait
ressembler le résultat. La colonne est encore du texte, mais du texte régulier :
`24/11/2011`, `24-11-2011`. Ses **quatre derniers caractères** sont donc
l'année, et ses **troisième et quatrième** le mois, quel que soit le séparateur.
Cela suffit à cadrer la période sans rien convertir.

In [ ]:
# .str[3:5] et .str[-4:] : on decoupe le texte par position
annee = propre["date"].str[-4:]    ## les 4 derniers caracteres
mois = propre["date"].str[3:5]     ## les 3e et 4e

print(annee.value_counts().to_dict())
print("mois presents en 2010 :", sorted(mois[annee == "2010"].unique()))

Deux années : 335 lignes en 2010 et 4 377 en 2011. Et en 2010, **un seul
mois** — décembre. Ce fichier couvre donc **de décembre 2010 à décembre
2011**, ce qui est cohérent avec l'historique d'un an qu'on nous a annoncé.

Retenez ce repère : c'est lui qui va nous permettre, dans deux cellules, de
repérer une conversion qui a échoué sans le dire.

**Temps 1 :** la façon naïve.

In [ ]:
# Cellule volontairement fausse : lisez le message d'erreur
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

`ValueError: time data "24-11-2011" doesn't match format "%d/%m/%Y"`

Le fichier mélange deux écritures : `14/11/2011` et `24-11-2011`. pandas veut
un format unique. Le message suggère lui-même la solution : `format="mixed"`.

**Temps 2 :** on ajoute `format="mixed"`.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## deux ecritures

print("date la plus ancienne :", essai.min())
print("date la plus recente  :", essai.max())

Plus d'erreur. Mais confrontez le résultat au repère du temps 0 : le fichier
va de décembre 2010 à décembre 2011, et pandas annonce **janvier 2010**.
**C'est faux.**

Pourquoi ? Parce que `01/12/2010` a été lu **à l'américaine** : mois d'abord,
donc le 12 janvier. En français, c'est le 1er décembre.

**Temps 3 :** on impose la lecture française avec `dayfirst=True`.

In [ ]:
# dayfirst=True : lecture francaise, le jour avant le mois
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)

print("date la plus ancienne :", propre["date"].min())
print("date la plus recente  :", propre["date"].max())

> ⚠️ **Le point le plus important de la séance.** L'étape 1 produisait une
> **erreur bruyante** : gênante, mais elle vous arrête. L'étape 2 produisait
> une **erreur silencieuse** : le code tourne, les chiffres s'affichent, et
> ils sont faux. C'est de très loin la plus dangereuse.
>
> Après toute conversion, **vérifiez que le résultat est plausible** :
> `.min()`, `.max()`, un `head()`. Trente secondes qui vous éviteront de
> présenter des chiffres faux.

Une fois la colonne convertie en date, `.dt` ouvre tout :

In [ ]:
propre["mois"] = propre["date"].dt.month           ## .dt = boite a outils
propre["jour_sem"] = propre["date"].dt.dayofweek   ## 0 = lundi, 6 = dimanche

propre[["date", "mois", "jour_sem"]].head(3)

---

### ✏️ À vous 4a — Les dates, correctement

> **Votre mission :**
> - Convertir `net['date']` en vraies dates.
> - Le fichier mélange `14/11/2011` et `24-11-2011`, et il est au format **français**.
> - Puis mettre la date la plus ancienne dans `date_min`.

In [ ]:
# format="mixed" : deux ecritures cohabitent dans la colonne
# dayfirst=True : format francais, le jour avant le mois
net["date"] = pd.to_datetime(net["date"], format="mixed", dayfirst=True)
date_min = net["date"].min()   ## min() sur des dates : la plus ancienne

print(date_min)

In [ ]:
verifier("4a - date la plus ancienne", str(date_min)[:10] == "2010-12-01",
         "sans dayfirst=True, 01/12/2010 est lu comme le 12 janvier")

---

### ✏️ À vous 4b — La fenêtre d'observation

> **Votre mission :**
> - Sur `net`, maintenant que les dates sont converties : la date la plus ancienne, la plus récente, et le nombre de **jours** entre les deux → `nb_jours`.
> - Un rapport annuel calculé sur cette période serait-il honnête ? Regardez la date de fin.
> - *Nouveau :* une soustraction de dates donne une durée ; `.days` en extrait le nombre de jours.

In [ ]:
debut = net["date"].min()
fin = net["date"].max()
nb_jours = (fin - debut).days   ## une duree, en jours

print("du", debut.date(), "au", fin.date(), ":", nb_jours, "jours")

# 373 jours : un peu plus d'un an, et surtout le fichier s'arrete le
# 9 decembre. Comparer decembre aux autres mois reviendrait a comparer
# neuf jours a trente. On y revient en seance 2.4.

In [ ]:
verifier("4b - duree couverte", nb_jours == 373,
         "soustrayez les deux dates, puis .days sur le resultat")

## Défaut 5 — Du texte incohérent

In [ ]:
print("nombre de categories distinctes :", propre["categorie"].nunique())
propre["categorie"].unique()[:8]

24 catégories, alors qu'il n'en existe que 8. Regardez bien : `' cuisine'`
avec un espace devant, `'CUISINE'` en majuscules, `'cuisine'`. Pour pandas,
ce sont **trois catégories différentes** — et un `value_counts()` sur cette
colonne éclaterait la cuisine en trois lignes, chacune sous-estimée.

> ⚠️ L'espace en début de chaîne est **invisible à l'écran**. C'est ce qui
> rend ce défaut particulièrement traître.

In [ ]:
# .str.strip() enleve les espaces au bord, .str.lower() met en minuscules
propre["categorie"] = propre["categorie"].str.strip().str.lower()   ## 24 -> 8

print("apres nettoyage :", propre["categorie"].nunique(), "categories")

## Défaut 6 — Les valeurs aberrantes

In [ ]:
propre["qte"].describe().round(1)   ## regarder min et max avant tout

Un minimum **négatif** et un maximum à **99 999**. Deux anomalies, mais elles
n'ont rien à voir :

- **`qte` négatif** : ce sont des **retours**. Ce n'est pas une erreur, c'est
  une information métier. On les écarte du calcul de chiffre d'affaires, mais
  on ne les jette pas — un taux de retour, ça s'analyse.
- **`qte = 99999`** : personne ne commande 99 999 articles. C'est une saisie
  erronée, ou un code sentinelle. On l'écarte.

> Traiter ces deux cas de la même façon serait une faute d'analyse.

In [ ]:
retours = propre.query("qte < 0")   ## un retour, pas une erreur
print("retours :", len(retours), "lignes")
print("quantites aberrantes :", len(propre.query("qte >= 10000")), "lignes")

avant = len(propre)
propre = propre.query("qte > 0 and qte < 10000").copy()   ## "and" dans query
print(avant, "->", len(propre))

---

### ✏️ À vous 5a — Uniformiser les catégories

> **Votre mission :**
> - Sur `net` : enlever les espaces autour et tout passer en minuscules.
> - Mettre le nombre de catégories restantes dans `nb_cat`.

In [ ]:
# strip() enleve les espaces en debut et fin (invisibles a l'ecran !)
# lower() met tout en minuscules
net["categorie"] = net["categorie"].str.strip().str.lower()   ## les deux !
nb_cat = net["categorie"].nunique()   ## 8, comme au catalogue

print(nb_cat)

In [ ]:
verifier("5a - categories uniformisees", nb_cat == 8,
         "il en reste beaucoup plus si vous n'avez fait que l'une des deux operations")

---

### ✏️ À vous 5b — Retours et aberrations

> **Votre mission :**
> - Sur `net` : compter les **retours** (`qte` strictement négatif) → `nb_retours`, et les quantités **aberrantes** (`qte` ≥ 10 000) → `nb_aberrants`.
> - Puis créer `final` : les quantités strictement positives et inférieures à 10 000.
> - Tout est à écrire. Rappel : dans `query()`, deux conditions se combinent avec `and`.

In [ ]:
# Un retour n'est pas une erreur : c'est une information metier,
# et un taux de retour, ca s'analyse. Une quantite de 99 999, en revanche,
# est une saisie erronee.
nb_retours = len(net.query("qte < 0"))          ## une info metier
nb_aberrants = len(net.query("qte >= 10000"))   ## une saisie erronee

final = net.query("qte > 0 and qte < 10000").copy()

print(nb_retours, "retours |", nb_aberrants, "aberrants |", len(final), "conservees")

In [ ]:
verifier("5b - retours", nb_retours == 107, "la condition est qte < 0")
verifier("5c - aberrants", nb_aberrants == 14, "la condition est qte >= 10000")
verifier("5d - lignes conservees", len(final) == 4591,
         "dans query() les conditions se combinent avec and")

## Enrichir — classer des lignes selon une règle

Une dernière commande, qui ne répare rien : elle **ajoute** une colonne. On
écrit la règle **une fois**, pandas l'applique à chaque ligne.

In [ ]:
propre["ca"] = propre["qte"] * propre["prix"]   ## calculable enfin

# np.where(condition, valeur_si_vrai, valeur_si_faux)
propre["type"] = np.where(propre["ca"] > 50, "grosse", "petite")   ## deux cas
propre["type"].value_counts()

> 💡 `np.where` ne tranche qu'entre **deux** possibilités. Pour trois catégories
> ou plus, et pour découper une colonne en tranches, deux commandes vous
> attendent dans la **feuille facultative** : `np.select` et `pd.cut`. Elles
> resserviront au bloc 3.

## Le bilan

Bonne pratique pour finir : un petit compte rendu de ce qu'on a retiré. Ça
tient en trois lignes, et ça évite d'avoir à se demander, trois semaines plus
tard, d'où viennent les lignes qui manquent.

---

### ✏️ À vous 6 — Le compte rendu

> **Votre mission :**
> - Dernier exercice, sur **votre** fichier : le **taux de perte** en pourcentage, arrondi à 1 décimale → `taux_perte`.
> - Formule : `100 * (1 - lignes_finales / lignes_initiales)`.
> - C'est le chiffre que vous présentez à votre responsable — pas « j'ai nettoyé les données ».

In [ ]:
# final = ce qu'on garde, sale = le fichier de depart
taux_perte = round(100 * (1 - len(final) / len(sale)), 1)   ## en %

print("perte :", taux_perte, "%")
print("dont 251 doublons, 407 sans client, 107 retours, 14 aberrants")

In [ ]:
verifier("6 - taux de perte", taux_perte == 14.5,
         "comparez le fichier final au fichier de depart sale")

14,5 % de pertes, et chacune s'explique. C'est plus utile à un lecteur que
« j'ai nettoyé les données ».

Le chiffre sert aussi de garde-fou : à 40 % de pertes, il vaudrait mieux
reprendre le pipeline depuis le début.

---

# Corrigé de la feuille

Les exercices de la séance sont corrigés plus haut, dans le fil du cours. Les cellules ci-dessous rejouent le setup pour rester exécutables isolément.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
sale = pd.read_csv(BASE + "ventes_sale.csv")   ## le fichier brut

# Le pipeline de la seance, rejoue d'un coup : c'est le point de depart
# des questions ci-dessous. Meme ordre qu'en seance.
net = sale.drop_duplicates().dropna(subset=["client_id"]).copy()
net["prix"] = pd.to_numeric(net["prix"].str.replace(" EUR", "")
                            .str.replace(",", "."))
net["date"] = pd.to_datetime(net["date"], format="mixed", dayfirst=True)
net["categorie"] = net["categorie"].str.strip().str.lower()

print(len(sale), "->", len(net))

### Exercice 1 — Extraire le mois

> **Votre mission :**
> - Créer la colonne `mois` de `net` à partir de `date`.
> - Mettre le numéro du mois qui compte le plus de lignes dans `mois_top`.

In [ ]:
# .dt donne acces aux composants d'une colonne de dates
net["mois"] = net["date"].dt.month   ## un entier de 1 a 12

# idxmax() renvoie l'etiquette (le numero du mois), pas l'effectif
mois_top = net["mois"].value_counts().idxmax()   ## octobre

print(mois_top)

In [ ]:
verifier("1 - mois le plus charge", mois_top == 10,
         ".dt.month sur une colonne de dates, puis value_counts().idxmax()")

### Exercice 2 — Classer selon plusieurs conditions — `np.select`

> **Votre mission :**
> - `np.where` ne tranche qu'entre deux cas. `np.select` en accepte autant qu'on veut.
> - Il attend trois choses : une **liste de conditions** (dans l'ordre où on veut qu'elles soient essayées), une **liste d'étiquettes** de même longueur, et `default=...` pour les lignes qu'aucune condition n'a retenues.
> - Créer `net['ca']`, puis une colonne `taille` : « tres grosse » au-dessus de 200 €, « grosse » au-dessus de 50 €, « petite » sinon. Compter les « tres grosse » → `nb_tres_grosses`.
> - ⚠️ **C'est la première condition vraie qui l'emporte** : de la plus restrictive à la plus large. Dans l'autre ordre, « tres grosse » resterait vide sans qu'aucune erreur ne le signale.

In [ ]:
net["ca"] = net["qte"] * net["prix"]

# De la condition la plus restrictive a la plus large : une ligne a 300 EUR
# est attrapee par la premiere et ne voit jamais la seconde.
conditions = [net["ca"] > 200, net["ca"] > 50]
etiquettes = ["tres grosse", "grosse"]

net["taille"] = np.select(conditions, etiquettes, default="petite")
nb_tres_grosses = (net["taille"] == "tres grosse").sum()

print(net["taille"].value_counts())

In [ ]:
verifier("2 - lignes tres grosses", nb_tres_grosses == 59,
         "conditions dans l'ordre 200 puis 50 : la premiere vraie l'emporte")

### Exercice 3 — Découper en tranches — `pd.cut`

> **Votre mission :**
> - Ranger les prix en quatre gammes : `entree` (0 à 1 €), `eco` (1 à 5 €), `milieu` (5 à 20 €), `premium` (au-delà).
> - Combien de références tombent en `entree` → `nb_entree` ?
> - *Nouveau :* `pd.cut(col, bins=[...], labels=[...])` — **cinq bornes** délimitent **quatre** tranches.

In [ ]:
# 5 bornes -> 4 tranches. Les etiquettes vont dans le meme ordre.
net["gamme"] = pd.cut(net["prix"],
                      bins=[0, 1, 5, 20, 10000],
                      labels=["entree", "eco", "milieu", "premium"])
nb_entree = (net["gamme"] == "entree").sum()

print(net["gamme"].value_counts())

In [ ]:
verifier("3 - references d'entree de gamme", nb_entree == 1093,
         "quatre etiquettes pour cinq bornes, dans l'ordre croissant")

### Question 4 — Le prix d'un `dropna()` négligent

> **Votre mission :**
> - Combien de lignes resteraient après un `dropna()` **sans argument** sur `sale` ?
> - Et après `dropna(subset=['client_id'])` ?
> - Ici les deux donnent le même résultat. Dans quel cas seraient-ils très différents ?

In [ ]:
print("depart              :", len(sale))
print("dropna() brut       :", len(sale.dropna()))   ## UN trou suffit a perdre
print("dropna(subset=...)  :", len(sale.dropna(subset=["client_id"])))

# Meme resultat ici, parce qu'une seule colonne a des trous. Sur un
# fichier ou trois colonnes en ont chacune 5 %, le dropna() brut en
# supprimerait 15 % : il suffit d'UN trou sur la ligne pour la perdre.
# Toujours nommer la colonne dont l'absence est redhibitoire.

### Question 5 — Les aberrations, sans seuil arbitraire

> **Votre mission :**
> - En séance, on a écarté `qte >= 10000` — un seuil choisi à la main.
> - Refaire le repérage avec la **règle de l'écart interquartile** : est aberrant ce qui dépasse `q3 + 1.5 * (q3 - q1)`.
> - Combien de lignes dépasse-t-elle ? Faut-il toutes les supprimer ?

In [ ]:
q1 = sale["qte"].quantile(0.25)   ## un quart des lignes en dessous
q3 = sale["qte"].quantile(0.75)   ## trois quarts en dessous
seuil = q3 + 1.5 * (q3 - q1)      ## la regle de l'ecart interquartile

print("seuil :", seuil, "->", (sale["qte"] > seuil).sum(), "lignes au-dessus")

# 379 lignes, dont seulement 17 a 99 999. Les autres montent jusqu'a
# 576 : ce sont de VRAIES grosses commandes de grossistes.
# La regle de l'IQR sert a REGARDER, pas a supprimer automatiquement.
# Supprimer les 379 reviendrait a jeter les meilleurs clients.

### Question 6 — Le compte rendu qualité

> **Votre mission :**
> - Vous rendez le fichier nettoyé. Produire les **quatre chiffres** de la note d'accompagnement :
> - lignes au départ · lignes conservées · taux de perte en % · **part du chiffre d'affaires réel** que représentent les ventes écartées faute de client identifié.
> - Ce dernier chiffre est celui qu'on oublie. Comparez-le au taux de perte en lignes : que vous dit l'écart entre les deux ?
> - ⚠️ Calculez ce CA sur les ventes **plausibles** uniquement. Les quantités à 99 999 produiraient sinon un total fictif qui écrase tout le reste.

In [ ]:
base = sale.drop_duplicates().copy()
base["prix"] = pd.to_numeric(base["prix"].astype(str)
                             .str.replace(",", ".").str.replace(" EUR", ""))
base["ca"] = base["qte"] * base["prix"]

# Les ventes plausibles : ni retours, ni saisies a 99 999
reel = base.query("qte > 0 and qte < 10000")
sans_client = reel[reel["client_id"].isna()]   ## celles qu'on ecarte
garde = reel.dropna(subset=["client_id"])      ## celles qu'on conserve

print(len(sale), "lignes au depart |", len(garde), "conservees")
print("perte de lignes :", round(100 * (1 - len(garde) / len(sale)), 1), "%")
print("CA ecarte       :", round(sans_client["ca"].sum(), 2), "euros, soit",
      round(100 * sans_client["ca"].sum() / reel["ca"].sum(), 1), "% du CA reel")

# 393 ventes bien reelles, 9 744 EUR, ecartees parce qu'on ne sait pas
# a QUI les attribuer. Elles pesent 8,0 % du CA pour 7,9 % des lignes :
# ce sont des ventes de taille ordinaire, pas un segment particulier.
# Si l'ecart entre les deux pourcentages avait ete grand, la perte
# aurait ete BIAISEE et il aurait fallu le signaler.

---

## Ce que vous savez faire maintenant

| Le problème | La commande |
|---|---|
| compter les lignes qui remplissent une condition | `(condition).sum()` |
| repérer les manquants | `df.isna().sum()` |
| supprimer les lignes incomplètes | `df.dropna(subset=["client_id"])` |
| remplacer les manquants | `df["prix"].fillna(0)` |
| compter les doublons | `df.duplicated().sum()` |
| supprimer les doublons | `df.drop_duplicates()` |
| texte → nombre | `pd.to_numeric(col, errors="coerce")` |
| texte → date | `pd.to_datetime(col, format="mixed", dayfirst=True)` |
| extraire le mois | `df["date"].dt.month` |
| nettoyer du texte | `col.str.strip().str.lower()` |
| enlever un morceau de texte | `col.str.replace(" EUR", "")` |
| classer selon une condition | `np.where(cond, "oui", "non")` |

Deux commandes vont plus loin et vous attendent dans la feuille facultative :
`np.select` (classer selon **plusieurs** conditions) et `pd.cut` (découper une
colonne de nombres en **tranches**).

## La règle d'or

**Le nettoyage est un pipeline, pas une série de bricolages.** Écrivez-le dans
l'ordre, de haut en bas, en repartant toujours du fichier brut. Le jour où on
vous livre le fichier du mois suivant, vous relancez le notebook et c'est fini.

Et **notez toujours combien de lignes vous perdez à chaque étape**. Un
nettoyage qui fait disparaître 40 % des données n'est pas un nettoyage, c'est
une erreur.